# Derived Kickbase points per 90 minutes × odds × lineup score

Historical points per 90 minutes = **90 × sum(weight × derived Kickbase points) / sum(weight × minutes played)**. Use team-match weights **28/24/20/16/12**, newest first. Missing appearances retain their original slots. Only appearances of **at least 10 minutes** contribute points and minutes; negative points are retained. Players without usable history receive score zero and a review reason.

Final score = **multiplier × points per 90 minutes × expected match points × starting chance**, with multiplier **1** by default.

**Goalkeepers:** expected match points use the compressed odds scale **1.1 to 1.7**. Alternative rank weights are **1, 0.60, 0.36, …**, using the same provider-specific slot normalization and injury handling as outfield players. Outfield decay remains **0.45**. Sources without alternatives retain their existing handling.

Run `09_sofascore_kickbase_point_averages.ipynb` first to generate a minute-bearing derived export; older exports must be regenerated. The corresponding team-form snapshot is required. Existing appearance eligibility is retained.

LigaInsider/Kickbase/Kicker/RotoWire weights default to **4/3/2/1**. The workflow retains source-matchday confirmations, persisted player-name mappings, review tables, and timestamped optimizer CSV exports. The export includes points per 90 minutes and individual source chances. Settings below are editable.


Goalkeeper expected match points are `1.1 + raw expected match points * (0.6 / 3)`: raw 0 maps to 1.1, 1.5 maps to 1.4, and 3 maps to 1.7. Outfield players retain raw 0–3 values. The exported `expected_match_points` is the value used in the score. Rerun this notebook to generate updated scores.

Goalkeeper alternatives use decay **0.60** (relative weights **1, 0.6, 0.36, …**), with existing provider normalization and injury handling. Outfield alternatives retain their configured decay.


In [ ]:
import sys
from pathlib import Path

# Absolute starting-chance reduction for a `QUES` injury flag from a lineup source.
QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY = 0.15
# Each rank is 45% of the preceding alternative before its slot is normalized.
ALTERNATIVE_STARTING_CHANCE_DECAY = 0.45
# Relative trust weights for active lineup sources; use 0 to disable one.
LINEUP_SOURCE_WEIGHTS = {'ligainsider': 4.0, 'kickbase': 3.0, 'kicker': 2.0, 'rotowire': 1.0}
# You will confirm whether each source is current for the selected matchday.
SCORE_MULTIPLIER = 1.0
GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY = 0.60

def locate_project_root() -> Path:
    starts = [Path.cwd().resolve()]
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.insert(0, Path(notebook_path).expanduser().resolve().parent)
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py.')

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import sofascore_kickbase_points_per_minute_odds_lineup_score as score_creation
import sofascore_rating_odds_lineup_score as shared_score_creation
importlib.reload(shared_score_creation)
importlib.reload(score_creation)  # pick up local score-formula changes without restarting the kernel
from sofascore_kickbase_points_per_minute_odds_lineup_score import run_score_creation
result = run_score_creation(
    questionable_injury_starting_chance_penalty=QUESTIONABLE_INJURY_STARTING_CHANCE_PENALTY,
    alternative_starting_chance_decay=ALTERNATIVE_STARTING_CHANCE_DECAY,
    lineup_source_weights=LINEUP_SOURCE_WEIGHTS,
    score_multiplier=SCORE_MULTIPLIER,
    goalkeeper_alternative_starting_chance_decay=GOALKEEPER_ALTERNATIVE_STARTING_CHANCE_DECAY,
)
